# CBSE Study Assistant — APK Builder
Upload `project_3.zip` and build. No local setup needed.

In [ ]:
import os, glob, shutil
from google.colab import files

!apt-get update -qq
!apt-get install -y -qq git zip unzip openjdk-17-jdk python3-pip \
    autoconf libtool pkg-config zlib1g-dev libncurses5-dev \
    libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev \
    python3-setuptools python3-dev 2>&1 | tail -3
!pip install -q buildozer cython 2>&1 | tail -3
print('Dependencies installed')

In [ ]:
# Upload zip
uploaded = files.upload()
zip_path = list(uploaded.keys())[0]
print('Uploaded:', zip_path)

In [ ]:
# Extract with system unzip (more reliable than Python zipfile on Colab)
EXTRACT_DIR = '/content/project'
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

!unzip -o "{zip_path}" -d "{EXTRACT_DIR}" 2>&1 | tail -5
print()
print('Extracted contents:')
!ls -la "{EXTRACT_DIR}"

In [ ]:
# Navigate into the project root (handles project_3/ prefix)
root = '/content/project'
for dirpath, dirnames, filenames in os.walk(root):
    if 'main.py' in filenames and 'buildozer.spec' in filenames:
        os.chdir(dirpath)
        print('Changed to:', dirpath)
        break
else:
    os.chdir(root)
    print('Changed to:', root)

print()
print('Working directory:', os.getcwd())
!ls -la

In [ ]:
# Verify required files
required = ['main.py', 'data_loader.py', 'recommender.py', 'cbse_chapters.csv', 'buildozer.spec']
all_ok = True
for f in required:
    ok = os.path.exists(f)
    print('[%s] %s' % ('OK' if ok else 'MISSING', f))
    all_ok = all_ok and ok

if os.path.exists('user_profile.json'):
    os.remove('user_profile.json')
    print('Removed user_profile.json')

if not all_ok:
    raise RuntimeError('Files missing. Check zip contents above.')
else:
    print('\nAll files present. Ready to build!')

## Build the APK
Takes **25–40 min** first time. Colab may disconnect — re-run Cells 2+5 to resume.

In [ ]:
# Clean cached build settings from previous failed runs
if os.path.exists('.buildozer'):
    import shutil
    shutil.rmtree('.buildozer')
    print('Removed stale .buildozer')

import subprocess
with open('build.log', 'w') as logf:
    p = subprocess.Popen('yes | buildozer android debug 2>&1', shell=True, stdout=logf, stderr=subprocess.STDOUT)
    p.wait()
print('Exit code:', p.returncode)
!tail -40 build.log

In [ ]:
# Search for the actual error in the full build log
import re
with open('build.log') as f:
    log = f.read()

# Find the p4a section and any error lines
lines = log.split('\n')
error_section = []
capture = False
for i, line in enumerate(lines):
    if '# Package the application' in line or '# Command failed' in line:
        capture = True
    if capture:
        error_section.append(f'{i+1}: {line}')
        if len(error_section) > 200:
            break

# Also search for Python tracebacks
traceback_lines = []
for i, line in enumerate(lines):
    if 'Traceback (most recent call last)' in line:
        start = max(0, i-2)
        for j in range(start, min(len(lines), i+40)):
            traceback_lines.append(f'{j+1}: {lines[j]}')
        traceback_lines.append('---')

if traceback_lines:
    print('=== PYTHON TRACEBACK ===')
    print('\n'.join(traceback_lines))
if error_section:
    print('=== BUILD END SECTION ===')
    print('\n'.join(error_section))
if not traceback_lines and not error_section:
    print('=== LAST 100 LINES (no traceback found) ===')
    print('\n'.join(lines[-100:]))

In [ ]:
# Download APK if successful
apk_files = glob.glob('bin/*.apk')
if apk_files:
    print('APK:', apk_files[0])
    files.download(apk_files[0])
else:
    print('No APK found in bin/')